# 🧪 ML Experiment Tracker – Quant Researcher
**Workflow**: Feature Load → Model Training → SHAP Interpretability → MLflow Logging → Model Comparison

---

In [ ]:
from qtrader.research import AnalystSession, RoleContext
from qtrader.features.store import FeatureStore
import polars as pl
import numpy as np
from scipy.special import erfinv
import plotly.express as px
import plotly.graph_objects as go
import shap
import matplotlib.pyplot as plt

session = AnalystSession(role=RoleContext.RESEARCHER)

SYMBOL   = 'BTC-USD'
TIMEFRAME = '1d'
TARGET   = 'fwd_ret_1'
EXPERIMENT = 'qtrader_alpha_research'
TEST_FRAC = 0.2

# Plotly Dark Theme Config
PLOTLY_DARK = dict(template="plotly_dark", plot_bgcolor='#0f1117', paper_bgcolor='#0f1117')

## 1. Feature Loading
Pulling pre-computed technical factors from the `FeatureStore`.

In [ ]:
df = session.load_features(SYMBOL, TIMEFRAME)

if df.is_empty():
    print("⚠️  FeatureStore empty. Generating data...")
    df = session.sample_ohlcv(symbol='BTC', days=365)
    df = session.make_returns(df)
    df = session.add_rolling_features(df)
    df = session.run_alpha_score(df, forward_periods=[1, 5])
    df = df.drop_nulls()

feature_cols = [c for c in df.columns if c.startswith(('sma_', 'vol_', 'rsi_', 'avg_'))]
print(f"Experimenting with {len(feature_cols)} features on {TARGET}")
df.head()

## 2. Walk-Forward Split
Maintaining time-series integrity by avoiding random splits.

In [ ]:
valid_df = df.select(feature_cols + [TARGET]).drop_nulls()
n = len(valid_df)
split = int(n * (1 - TEST_FRAC))
train_df = valid_df.head(split)
test_df  = valid_df.tail(n - split)

X_train, y_train = train_df.select(feature_cols).to_numpy(), train_df[TARGET].to_numpy()
X_test, y_test   = test_df.select(feature_cols).to_numpy(), test_df[TARGET].to_numpy()
print(f"Train subset: {X_train.shape} | Test subset: {X_test.shape}")

## 3. Training & MLflow Tracking
Logging architecture, hyperparams, and Information Coefficient (IC).

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

mlflow.set_experiment(EXPERIMENT)

params = {'n_estimators': 150, 'max_depth': 5, 'min_samples_leaf': 10}

with mlflow.start_run(run_name=f"{SYMBOL}_RF_Alpha") as run:
    model = RandomForestRegressor(**params, random_state=42)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    ic = float(np.corrcoef(preds, y_test)[0, 1])

    mlflow.log_params(params)
    mlflow.log_metrics({'MAE': mae, 'IC': ic})
    mlflow.sklearn.log_model(model, "model")

    print(f"Training Complete. IC: {ic:.4f} | MAE: {mae:.6f}")

## 4. SHAP Interpretability
Understanding which features drive the model's predictions.

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

print("Global Feature Importance via SHAP:")
shap.summary_plot(shap_values, X_test, feature_names=feature_cols, plot_type="bar", show=False)
plt.gcf().set_facecolor('#0f1117')
plt.gca().set_facecolor('#0f1117')
plt.show()

## 5. Model Analytics
Visualizing prediction vs reality and historical run performance.

In [ ]:
# 1. Prediction Scatter
fig_scat = px.scatter(
    x=y_test, y=preds,
    labels={'x': 'Actual Returns', 'y': 'Predicted Alpha'},
    title="Actual vs Predicted Return (Test Set)",
    trendline="ols",
    trendline_color_override="#ef4444"
)
fig_scat.update_layout(**PLOTLY_DARK)
fig_scat.show()

# 2. Historical IC Comparison
runs = mlflow.search_runs(experiment_names=[EXPERIMENT])
if not runs.empty:
    fig_runs = px.bar(
        runs.sort_values('metrics.IC', ascending=False).head(10),
        x='run_id', y='metrics.IC',
        title="Experiment Leaderboard: Information Coefficient (IC)",
        color='metrics.IC', color_continuous_scale='Viridis'
    )
    fig_runs.update_layout(**PLOTLY_DARK, xaxis_tickangle=-45)
    fig_runs.show()